In [0]:
%run ../utils/init.py

In [0]:
df_bronze = spark.read.format("delta").load(f"{BRONZE}/osm_poi/")
print(f"Lignes bronze : {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
%pip install h3

In [0]:
import h3
from pyspark.sql.types import StringType

# UDF H3 : coordonnées → index hexagonal résolution 9 (~175m)
@F.udf(StringType())
def to_h3(lat, lon):
    if lat is None or lon is None:
        return None
    return h3.latlng_to_cell(float(lat), float(lon), 9)

# Charger le référentiel communes
df_communes = spark.read.format("delta").load(f"{BRONZE}/referentiel_communes/")

# Générer les index H3 pour chaque commune (centroïde)
df_communes_h3 = df_communes \
    .withColumn("h3_index", to_h3(F.col("centroid_lat"), F.col("centroid_lon"))) \
    .select("h3_index", "code_commune", "nom_commune", "code_dept") \
    .withColumnRenamed("nom_commune", "city") \
    .withColumnRenamed("code_dept", "code_departement")

# Générer les index H3 pour chaque POI
df_poi_h3 = df_bronze.withColumn("h3_index", to_h3(F.col("latitude"), F.col("longitude")))

# Jointure H3
df_silver = (df_poi_h3
    .join(df_communes_h3, "h3_index", "left")
    .filter(F.col("code_commune").isNotNull())
    .dropDuplicates(["osm_id"])
    .select(
        "osm_id", "osm_type", "name", "category",
        "amenity", "shop", "leisure",
        "latitude", "longitude",
        "is_bio_bobo", "h3_index",
        "code_commune", "city", "code_departement"
    )
)

print(f"Lignes silver : {df_silver.count()}")
df_silver.show(3)

In [0]:
df_communes_check = spark.read.format("delta").load(f"{BRONZE}/referentiel_communes/")
df_communes_check.printSchema()

In [0]:
%pip install geopandas shapely

In [0]:
import geopandas as gpd
import pandas as pd
from shapely import wkt
from shapely.geometry import Point

# Charger communes en pandas
communes_pd = df_communes_check.select(
    "code_commune", "nom_commune", "code_dept", "geometry_wkt"
).toPandas()

communes_pd["geometry"] = communes_pd["geometry_wkt"].apply(wkt.loads)
gdf_communes = gpd.GeoDataFrame(communes_pd, geometry="geometry", crs="EPSG:4326")

# Charger POI en pandas
poi_pd = df_bronze.select(
    "osm_id", "osm_type", "name", "category",
    "amenity", "shop", "leisure",
    "latitude", "longitude", "is_bio_bobo"
).toPandas()

poi_pd["geometry"] = poi_pd.apply(
    lambda r: Point(r["longitude"], r["latitude"]), axis=1
)
gdf_poi = gpd.GeoDataFrame(poi_pd, geometry="geometry", crs="EPSG:4326")

# Spatial join POI → Commune
gdf_joined = gpd.sjoin(gdf_poi, gdf_communes, how="left", predicate="within")

print(f"POI rattachés à une commune : {gdf_joined['code_commune'].notna().sum()}")
print(f"Total POI : {len(gdf_joined)}")

In [0]:
# Nettoyage et conversion Spark
gdf_result = gdf_joined[gdf_joined["code_commune"].notna()][[
    "osm_id", "osm_type", "name", "category",
    "amenity", "shop", "leisure",
    "latitude", "longitude", "is_bio_bobo",
    "code_commune", "nom_commune", "code_dept"
]].rename(columns={
    "nom_commune": "city",
    "code_dept": "code_departement"
})

df_silver = spark.createDataFrame(gdf_result)

print(f"Lignes silver : {df_silver.count()}")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("category", "code_departement") \
    .save(f"{SILVER}/osm_poi/")

print("✅ Silver OSM POI écrit")